[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/npv_decisions.ipynb)

# Capital budgeting full course: the practice set, in Python

Thirty-six exercises in four tiers, the same as the practice spreadsheet: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (net present value, payback, the book rate of return, the internal rate of return, a
limited budget, cash flows after tax, equivalent annual cost), **tier 3** stretch problems (combine ideas, spot the
trap, run a rule backwards) and **tier 4**, the case: Bangalore Cold Chain (a fictional company; our own data) has two
designs for a new automated warehouse, and has to decide which one adds more value. Each tier is harder than the
last.

Under every tier there's a short **how you'd do this in Python** cell: the same cash flows valued in plain Python,
with `numpy` for arrays of cash flows and `pandas` for year-by-year tables, so you can run them on your own
forecasts. And four cells you can move yourself: the cost of capital sliding along a project's NPV curve to the rate
where it crosses zero, a project whose curve crosses twice, two projects whose curves cross each other, and the
case's two designs.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When a slider appears, move it and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no
install) and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Money in
dollars, to the cent: `16872.77`. Rates as percentages: `17.80` for 17.80 % (typing the % sign is fine, and so is the
rate itself, `0.178`). Whole years as a number: `3`. The case's values in $ millions, to two decimals: `9.18`.
Numbers without thousands separators: `16872.77`, not `16,872.77`. You can type the arithmetic too:
`-5000+3000/1.09+3000/1.09**2`. For several answers, separate them with commas, in the order the question lists them
(the box's label shows the order). You can also put your answer between the quotes in a cell's last line,
`practice("P1.2", "")`, and run the cell: it checks straight away.

**The cheat sheet.** Every rule and spreadsheet function from the course on two pages:
[NPV and capital budgeting cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/npv_decisions_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import ast, math, operator, re, textwrap
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # cash in chalk, chalk smudge, ghost lines
SKY, VIOLET, CORAL, GOLD, MINT = "#6fb6bb", "#b49ad6", "#e3896b", "#e8c56b", "#a0c27c"
# cash as it is paid CHALK · money moved to today SKY · the rate VIOLET · growth MINT · a shortcut that misleads CORAL ·
# the one thing to look at GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="our own example · computed"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}
FUNCS = {"sqrt": math.sqrt, "exp": math.exp, "ln": math.log, "log": math.log}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets and sqrt(…), exp(…), ln(…)."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") in FUNCS and len(n.args) == 1:
            return FUNCS[n.func.id](ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNITS = r"\b(dollars?|years?|months?|payments?|millions?|percent)\b|[$%]"

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    t = re.sub(UNITS, "", s, flags=re.I).strip()
    try: return calc(t)
    except Exception: return " ".join(s.lower().split())

def right(got, want, tol, pct=False):
    """Within the tolerance; a rate asked for in percent may also be typed as the rate itself (0.178 for 17.80 %)."""
    if isinstance(got, str): return False
    tol = max(tol, 1e-7)
    return abs(got - want) <= tol or (pct and abs(got * 100 - want) <= tol)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the same questions and answers as the spreadsheet) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Why NPV',
     "q": 'What is $20,000 received in 3 years worth today at 9 %?',
     "parts": ['present value ($)'],
     "answer": [15443.669601221283], "tol": [0.005], "pct": [], "function": 'PV',
     "working": '20,000/1.09³ = 20,000/1.295029 = 15,443.67'},
    {"id": 'P1.2', "tier": 1, "group": 'Why NPV',
     "q": 'A project costs $5,000 today and pays $3,000 at the end of each of the next two years. At 9 %, what is its NPV?',
     "parts": ['NPV ($)'],
     "answer": [277.3335577813309], "tol": [0.005], "pct": [], "function": 'NPV',
     "working": '−5,000 + 3,000/1.09 + 3,000/1.09² = −5,000 + 2,752.29 + 2,525.04 = 277.33'},
    {"id": 'P1.3', "tier": 1, "group": 'Payback and the book rate of return',
     "q": 'A project costs $12,000 and returns $4,000 at the end of each year for 5 years. What is its payback period, in years?',
     "parts": ['payback period (years)'],
     "answer": [3], "tol": [0], "pct": [], "function": 'SUM',
     "working": '4,000 + 4,000 + 4,000 = 12,000: the cost is back at the end of year 3'},
    {"id": 'P1.4', "tier": 1, "group": 'Payback and the book rate of return',
     "q": 'A $60,000 machine is depreciated straight-line to zero over 3 years and brings in $26,000 of cash a year. What is its book rate of return (average book income ÷ average book value), in percent?',
     "parts": ['book rate of return (%)'],
     "answer": [20.0], "tol": [0.005], "pct": [0], "function": 'AVERAGE',
     "working": 'Depreciation 20,000 a year; book income 26,000 − 20,000 = 6,000; average book value 60,000/2 = 30,000; 6,000/30,000 = 20 %'},
    {"id": 'P1.5', "tier": 1, "group": 'The internal rate of return',
     "q": 'Pay $40,000 today and receive $46,000 in one year. What is the internal rate of return, in percent?',
     "parts": ['internal rate of return (%)'],
     "answer": [14.99999999999998], "tol": [0.005], "pct": [0], "function": 'IRR',
     "working": '46,000/40,000 − 1 = 15 %: at 15 % the NPV is −40,000 + 46,000/1.15 = 0'},
    {"id": 'P1.6', "tier": 1, "group": 'The internal rate of return',
     "q": 'A customer pays you $10,000 today for work that will cost you $11,500 to deliver next year. Is this like lending (type 1) or like borrowing (type 2)?',
     "parts": ['lending (1) or borrowing (2)'],
     "answer": [2], "tol": [0], "pct": [], "function": '—',
     "working": 'Cash comes in first and goes out later: borrowing, at 15 %. It is worth doing only if 15 % is less than the opportunity cost of capital'},
    {"id": 'P1.7', "tier": 1, "group": 'Choosing between projects, and limited capital',
     "q": 'A project needs $80,000 of a limited budget and has an NPV of $20,000. What is its profitability index (two decimals)?',
     "parts": ['profitability index (two decimals)'],
     "answer": [0.25], "tol": [0.005], "pct": [], "function": '—',
     "working": 'NPV ÷ investment = 20,000/80,000 = 0.25'},
    {"id": 'P1.8', "tier": 1, "group": "Building a project's cash flows",
     "q": 'Revenue $200,000, cash costs $140,000, depreciation $30,000, tax rate 25 %. What is the operating cash flow after tax?',
     "parts": ['operating cash flow after tax ($)'],
     "answer": [52500.0], "tol": [0.005], "pct": [], "function": '—',
     "working": 'Taxable profit 30,000, tax 7,500, cash 200,000 − 140,000 − 7,500 = 52,500. Or (200,000 − 140,000) × 0.75 + 0.25 × 30,000 = 45,000 + 7,500'},
    {"id": 'P1.9', "tier": 1, "group": "Building a project's cash flows",
     "q": 'The nominal rate is 9 % and expected inflation 3 %. What is the real rate, in percent (exact, not the shortcut)?',
     "parts": ['real rate (%)'],
     "answer": [5.825242718446599], "tol": [0.005], "pct": [0], "function": '—',
     "working": '1.09/1.03 − 1 = 5.8252 % (the shortcut 9 − 3 = 6 % is a little high)'},
    {"id": 'P1.10', "tier": 1, "group": 'Lives, replacement and capacity',
     "q": "A machine costs $20,000, lasts 4 years and costs $3,000 a year to run (today's dollars). At a real rate of 6 %, what is its equivalent annual cost?",
     "parts": ['equivalent annual cost ($)'],
     "answer": [8771.829847465464], "tol": [0.005], "pct": [], "function": 'PMT',
     "working": '20,000/3.4651 + 3,000 = 5,771.83 + 3,000 = 8,771.83. In a spreadsheet: =-PMT(6%, 4, 20000, 0, 0) + 3000'},
    {"id": 'P2.1', "tier": 2, "group": 'Why NPV',
     "q": 'A cold-room upgrade costs $45,000 today and saves $15,000, $18,000 and $22,000 at the ends of years 1, 2 and 3. At 9 %, what is its NPV?',
     "parts": ['NPV ($)'],
     "answer": [899.7443300497471], "tol": [0.005], "pct": [], "function": 'NPV',
     "working": 'PVs 13,761.47 + 15,150.24 + 16,988.04 = 45,899.74; NPV = 899.74. In a spreadsheet: =-45000 + NPV(9%, 15000, 18000, 22000)'},
    {"id": 'P2.2', "tier": 2, "group": 'Why NPV',
     "q": 'Same upgrade: what does =NPV(9%, -45000, 15000, 18000, 22000) give, and why is it wrong?',
     "parts": ['what the one-cell NPV returns ($)'],
     "answer": [825.4535138071078], "tol": [0.005], "pct": [], "function": 'NPV',
     "working": "825.45 = the right NPV ÷ 1.09: the function treats its first value as one period away, so today's cost is pushed a year into the future. Add today's cash flow outside the function"},
    {"id": 'P2.3', "tier": 2, "group": 'Payback and the book rate of return',
     "q": 'A project costs $50,000 and returns $18,000 a year for 4 years. Give its payback period, and its discounted payback period at 9 % (whole years).',
     "parts": ['payback period (years)', 'discounted payback period (years)'],
     "answer": [3, 4], "tol": [0, 0], "pct": [], "function": 'SUM, NPV',
     "working": 'Payback: 18,000 × 3 = 54,000 ≥ 50,000: 3 years. Discounted: PVs 16,514, 15,150, 13,899 add to 45,563 < 50,000; with year 4, 58,315: 4 years'},
    {"id": 'P2.4', "tier": 2, "group": 'Payback and the book rate of return',
     "q": 'A $90,000 conveyor is depreciated straight-line over 3 years and returns $40,000, $38,000 and $36,000. Give its book rate of return in percent, and its NPV at 9 %.',
     "parts": ['book rate of return (%)', 'NPV at 9 % ($)'],
     "answer": [17.77777777777778, 6479.692732749601], "tol": [0.005, 0.005], "pct": [0], "function": 'AVERAGE, NPV',
     "working": 'Book income 10,000, 8,000, 6,000: average 8,000; average book value 45,000; 17.78 %. NPV = 6,479.69. The book return says nothing about the 9 % the investors give up'},
    {"id": 'P2.5', "tier": 2, "group": 'The internal rate of return',
     "q": 'A project costs $30,000 and pays $13,000 a year for 3 years. What is its IRR, in percent? Should it go ahead at a 9 % cost of capital?',
     "parts": ['IRR (%)'],
     "answer": [14.359667846177404], "tol": [0.005], "pct": [0], "function": 'IRR',
     "working": 'Try 14 %: NPV 181.22; 15 %: -318.07; IRR ≈ 14.36 % > 9 %: yes (NPV at 9 % = 2,906.83). =IRR(range)'},
    {"id": 'P2.6', "tier": 2, "group": 'The internal rate of return',
     "q": 'A pop-up store costs $20,000 to set up, brings in $47,000 in year 1 and costs $27,300 to remove in year 2. Find both rates at which its NPV is zero (percent), and its NPV at 9 %.',
     "parts": ['the lower rate (%)', 'the higher rate (%)', 'NPV at 9 % ($)'],
     "answer": [4.999999999999993, 29.99999999999995, 141.40223886878084], "tol": [0.005, 0.005, 0.005], "pct": [0, 1], "function": 'IRR, NPV',
     "working": '−20,000(1 + r)² + 47,000(1 + r) − 27,300 = 0 gives 1 + r = 1.05 or 1.30: 5 % and 30 %. NPV at 9 % = 141.40: positive, because 9 % lies between the two rates. =IRR(range, 0%) and =IRR(range, 40%)'},
    {"id": 'P2.7', "tier": 2, "group": 'Choosing between projects, and limited capital',
     "q": 'Only one of two projects can be done. One costs $20,000 and pays $27,000 in a year (35 %); the other costs $60,000 and pays $76,800 (28 %). At 9 %, give each NPV, and the IRR of the extra $40,000 (percent).',
     "parts": ["the smaller project's NPV ($)", "the bigger project's NPV ($)", 'IRR of the extra $40,000 (%)'],
     "answer": [4770.64220183486, 10458.715596330265, 24.5], "tol": [0.005, 0.005, 0.005], "pct": [2], "function": 'NPV, IRR',
     "working": 'NPVs 4,770.64 and 10,458.72: the bigger project adds more value. The extra project is −40,000 then +49,800: 24.50 % > 9 %, so the extra money is worth spending'},
    {"id": 'P2.8', "tier": 2, "group": 'Choosing between projects, and limited capital',
     "q": 'A budget of $150,000 must be shared among five projects (investment, NPV): (60,000, 15,000), (50,000, 14,000), (40,000, 12,000), (90,000, 20,000), (50,000, 9,000). Rank by profitability index. What is the most NPV the budget can buy?',
     "parts": ['the most NPV the budget can buy ($)'],
     "answer": [41000.0], "tol": [0.005], "pct": [], "function": 'SUMPRODUCT',
     "working": 'Indexes 0.25, 0.28, 0.30, 0.22, 0.18: take the 0.30, the 0.28 and the 0.25, which use exactly 150,000: NPV 12,000 + 14,000 + 15,000 = 41,000 (the two biggest NPVs alone would give 35,000)'},
    {"id": 'P2.9', "tier": 2, "group": "Building a project's cash flows",
     "q": 'A machine costs $80,000, is depreciated straight-line to zero over 4 years and saves $34,000 a year before tax. Tax is 25 %. What is its NPV at 9 %?',
     "parts": ['NPV ($)'],
     "answer": [18811.456250127885], "tol": [0.005], "pct": [], "function": 'NPV',
     "working": 'Cash flow 34,000 × 0.75 + 0.25 × 20,000 = 25,500 + 5,000 = 30,500 a year; × 3.2397 = 98,811.46; NPV = 18,811.46'},
    {"id": 'P2.10', "tier": 2, "group": 'Lives, replacement and capacity',
     "q": 'Machine one costs $24,000, lasts 3 years and costs $5,000 a year to run. Machine two costs $36,000, lasts 5 years and costs $4,500 a year. At a real rate of 6 %, give each equivalent annual cost.',
     "parts": ["machine one's equivalent annual cost ($)", "machine two's equivalent annual cost ($)"],
     "answer": [13978.63550697323, 13046.27041552282], "tol": [0.005, 0.005], "pct": [], "function": 'PMT',
     "working": '24,000/2.6730 + 5,000 = 13,978.64; 36,000/4.2124 + 4,500 = 13,046.27: machine two is cheaper per year'},
    {"id": 'P3.1', "tier": 3, "group": 'Why NPV',
     "q": 'A project will pay $12,000 a year for 5 years. What is the most a company should pay for it at 9 %? And if investments this risky are expected to earn 12 %?',
     "parts": ['the most to pay at 9 % ($)', 'the most to pay at 12 % ($)'],
     "answer": [46675.815160220634, 43257.31442814009], "tol": [0.005, 0.005], "pct": [], "function": 'PV',
     "working": 'The price at which NPV is zero is the present value of what you get: 46,675.82 at 9 %; 43,257.31 at 12 %. A riskier project is worth less today'},
    {"id": 'P3.2', "tier": 3, "group": 'Payback and the book rate of return',
     "q": 'A firm accepts projects that pay back within 3 years. Project one costs $100,000 and pays $40,000 a year for 3 years; project two costs $100,000 and pays $25,000 a year for 8 years. Give each NPV at 9 %. Which does the rule accept?',
     "parts": ["project one's NPV ($)", "project two's NPV ($)"],
     "answer": [1251.7866395269775, 38370.47786867543], "tol": [0.005, 0.005], "pct": [], "function": 'NPV',
     "working": 'NPVs 1,251.79 and 38,370.48. The rule accepts only project one (payback 3 years; project two takes 4) and turns away almost all the value: it ignores every cash flow after the cutoff'},
    {"id": 'P3.3', "tier": 3, "group": 'The internal rate of return',
     "q": 'A line costs $150,000 and returns $40,000, $60,000, $80,000 and $30,000. Find its IRR (percent) and its NPV at 9 %. By how many points could the cost of capital rise before the project stops adding value?',
     "parts": ['IRR (%)', 'NPV at 9 % ($)'],
     "answer": [14.969013623460025, 20225.482039256633], "tol": [0.005, 0.005], "pct": [0], "function": 'IRR, NPV',
     "working": 'IRR ≈ 14.97 %; NPV at 9 % = 20,225.48. The margin is 5.97 points: the IRR is the highest cost of capital the project can bear'},
    {"id": 'P3.4', "tier": 3, "group": 'The internal rate of return',
     "q": 'A supplier offers $80,000 today if you agree to pay it $30,000 a year for 3 years. Find the IRR (percent) and the NPV at 9 %. Should you accept?',
     "parts": ['IRR (%)', 'NPV at 9 % ($)'],
     "answer": [6.128560606098443, 4061.160020354764], "tol": [0.005, 0.005], "pct": [0], "function": 'IRR, NPV',
     "working": 'IRR ≈ 6.13 %, NPV = 4,061.16: accept. This is borrowing, so a rate BELOW the cost of capital is good news; the usual IRR rule would have said no'},
    {"id": 'P3.5', "tier": 3, "group": 'Choosing between projects, and limited capital',
     "q": 'Two projects each cost $100,000 and only one can be done. The fast one pays $70,000, $40,000, $20,000; the long one $10,000, $30,000, $110,000. At what cost of capital (percent) do their NPVs tie, and by how much does the long one lead at 9 %?',
     "parts": ['the cost of capital at which they tie (%)', "the long project's lead at 9 % ($)"],
     "answer": [14.424332188801959, 6033.8417131971455], "tol": [0.005, 0.005], "pct": [0], "function": 'IRR, NPV',
     "working": "The difference (long − fast) is −60,000, −10,000, +90,000: its IRR is 14.42 %. Below that rate the long project wins; at 9 % it leads by 6,033.84, though its IRR (16.6 %) is lower than the fast one's (18.2 %)"},
    {"id": 'P3.6', "tier": 3, "group": 'Choosing between projects, and limited capital',
     "q": 'A $100,000 budget, three projects (investment, NPV): (60,000, 18,000), (50,000, 14,000), (50,000, 13,000). What NPV does ranking by profitability index give, and what is the best the budget can do?',
     "parts": ['NPV from ranking by index ($)', 'the best the budget can do ($)'],
     "answer": [18000.0, 27000.0], "tol": [0.005, 0.005], "pct": [], "function": '—',
     "working": 'Indexes 0.30, 0.28, 0.26: the first takes 60,000, and neither of the others fits in the 40,000 left: 18,000. The other two together use the whole budget for 27,000. The index works only when the chosen projects fill the budget'},
    {"id": 'P3.7', "tier": 3, "group": "Building a project's cash flows",
     "q": "A project saves $80,000 a year for 5 years, forecast in today's dollars; costs will rise with 3 % inflation. The cost of capital is 9 % nominal. Value the savings correctly, then show what discounting the today's-dollar forecast at 9 % gives.",
     "parts": ['valued correctly, at the real rate ($)', "the mix-up: today's dollars at 9 % ($)"],
     "answer": [338597.24747224525, 311172.10106813756], "tol": [0.005, 0.005], "pct": [], "function": 'PV',
     "working": 'Real rate 5.825 %: 338,597.25 (the same as growing the savings 3 % a year and discounting at 9 %). Mixed: 311,172.10, too low: inflation is in the rate but not in the cash'},
    {"id": 'P3.8', "tier": 3, "group": 'Lives, replacement and capacity',
     "q": "An old machine will cost $11,000 to run next year, then $14,000, then $18,000. A new one costs $50,000, lasts 7 years and costs $4,000 a year (real rate 6 %). Find the new machine's equivalent annual cost. For how many more years should the old one be kept?",
     "parts": ["the new machine's equivalent annual cost ($)", 'more years to keep the old one'],
     "answer": [12956.750902950536, 1], "tol": [0.005, 0], "pct": [], "function": 'PMT',
     "working": '50,000/5.5824 + 4,000 = 12,956.75 a year. The old machine is cheaper than that next year (11,000) and dearer the year after (14,000): keep it one more year, then replace'},
    {"id": 'P4.1', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": "Four items are on the table: the $1.5 million design study (already paid), the $0.6 million a year the site's tenant pays, $1.0 million a year of margin the old warehouse would lose, and $1.0 million a year of interest on the bank loan. How much of this is a yearly cost of the full design, before tax, in today's dollars?",
     "parts": ['the yearly cost that counts, before tax ($ millions)'],
     "answer": [1.6], "tol": [0.005], "pct": [], "function": '—',
     "working": '1.6: the lost rent (an opportunity cost) and the lost margin (a side effect). The study is sunk, and the interest belongs to the financing decision: its cost is already in the 9 %'},
    {"id": 'P4.2', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'For year 1 of the full design, give the cash flow before tax, the tax, and the operating cash flow.',
     "parts": ['cash flow before tax, year 1 ($ millions)', 'tax, year 1 ($ millions)', 'operating cash flow, year 1 ($ millions)'],
     "answer": [9.682, 0.9205000000000005, 8.761500000000002], "tol": [0.005, 0.005, 0.005], "pct": [], "function": '—',
     "working": '(17.0 − 6.0 − 1.0 − 0.6) × 1.03 = 9.682; depreciation 36/6 = 6.0; tax 0.25 × (9.682 − 6.0) = 0.9205; operating cash flow 9.682 − 0.9205 = 8.7615 (or 9.682 × 0.75 + 1.5)'},
    {"id": 'P4.3', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": "Working capital is $1.0 million at the start, a tenth of each year's revenue in years 1 to 5, and zero at the end of year 6. Give the cash flow from working capital in year 1, and the present value of all its cash flows at 9 %.",
     "parts": ['cash flow from working capital, year 1 ($ millions)', 'present value of all its cash flows ($ millions)'],
     "answer": [-0.7510000000000003, -0.6766671887987794], "tol": [0.005, 0.005], "pct": [], "function": 'NPV',
     "working": 'Year 1: the level rises from 1.0 to 1.751, a cash flow of -0.751. Every dollar comes back in year 6 (1.971), yet the present value is -0.6767: tied-up money has a cost'},
    {"id": 'P4.4', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": "Find the full design's NPV at 9 % and its IRR (percent).",
     "parts": ["the full design's NPV ($ millions)", "the full design's IRR (%)"],
     "answer": [9.182358671658857, 15.981676081671592], "tol": [0.005, 0.005], "pct": [1], "function": 'NPV, IRR',
     "working": 'Cash flows -37.00, 8.01, 8.93, 9.15, 9.38, 9.62, 19.05: NPV 9.1824, IRR 15.98 %. =B20+NPV(9%, C20:H20) and =IRR(B20:H20)'},
    {"id": 'P4.5', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": "Do the same for the compact design, and give each design's payback period in years (full, then compact).",
     "parts": ["the compact design's NPV ($ millions)", "the compact design's IRR (%)", "the full design's payback (years)", "the compact design's payback (years)"],
     "answer": [6.47042061978822, 23.09466530983467, 5, 4], "tol": [0.005, 0.005, 0, 0], "pct": [1], "function": 'NPV, IRR',
     "working": 'Compact: NPV 6.4704, IRR 23.09 %; payback 5 years for the full design and 4 for the compact. The compact design wins on IRR and on payback, and adds less value'},
    {"id": 'P4.6', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": "Subtract the compact design's cash flows from the full design's. Find the NPV of that extra project at 9 % and its IRR (percent).",
     "parts": ["the extra project's NPV ($ millions)", "the extra project's IRR (%)"],
     "answer": [2.7119380518706393, 12.173062005816682], "tol": [0.005, 0.005], "pct": [1], "function": 'NPV, IRR',
     "working": 'The extra 24.4 million earns 12.17 %, above 9 %: NPV 2.7119. So the full design is the better choice while the cost of capital stays below that rate'},
    {"id": 'P4.7', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": "Value the full design in real terms (every cash flow in today's dollars, at the real rate). Then give the NPV if the forecast is left in today's prices and discounted at 9 %.",
     "parts": ['in real terms, at the real rate ($ millions)', "the mix-up: today's prices at 9 % ($ millions)"],
     "answer": [9.182358671658868, 5.303660528582046], "tol": [0.005, 0.005], "pct": [], "function": 'NPV',
     "working": 'Real rate 5.825 %: 9.1824, the same as nominal. Mixed: 5.3037: inflation in the rate but not in the cash hides a large part of the value'},
    {"id": 'P4.8', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": "The cooling plant: the long-life type costs $4.0 million, lasts 10 years and costs $0.25 million a year to run; the short-life type costs $2.4 million, lasts 5 years and costs $0.35 million a year (after tax, today's dollars). Give each equivalent annual cost at the real rate, then write the recommendation in two sentences.",
     "parts": ['long-life plant: equivalent annual cost ($ millions)', 'short-life plant: equivalent annual cost ($ millions)'],
     "answer": [0.7889810719658445, 0.9170453656470973], "tol": [0.005, 0.005], "pct": [], "function": 'PMT',
     "working": '4.0/7.4214 + 0.25 = 0.7890; 2.4/4.2325 + 0.35 = 0.9170. Build the full design (NPV 9.18 million, 2.71 more than the compact design) with the long-life plant, which costs 0.13 million a year less to own'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch", 4: "Case"}
assert len(EX) == 36 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g, w, t, i in q["pct"]) for i, (g, w, t) in enumerate(zip(got, q["answer"], q["tol"]))]
            if all(ok): print("✓ right" if n == 1 else f"✓ all {n} right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            if q["function"] not in ("—", ""): print(textwrap.fill(f"In a spreadsheet: {q['function']}.", 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers practice.py gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(float(g), w, t) for g, w, t in zip(got, answers(k), Q[k]["tol"]))

# the checker itself: a right answer passes, a wrong one doesn't; a rate may be typed as a percentage or as itself
assert right(read_part("-5000+3000/1.09+3000/1.09**2"), answers("P1.2")[0], 0.005)
assert not right(read_part("277"), answers("P1.2")[0], 0.005)
assert right(read_part("15%"), 15, 0.005, True) and right(read_part("0.15"), 15, 0.005, True)
assert not right(read_part("0.15"), 15, 0.005) and not right(read_part("fifteen"), 15, 0.005, True)

## Tier 1 · Warm-up

One idea, one step, small numbers: a present value, a net present value, a payback period, a book rate of return, an
internal rate of return, lending or borrowing, a profitability index, an operating cash flow after tax, a real rate,
and an equivalent annual cost.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### The sorting line in three lines

The course's sorting line costs $100,000 today and returns $50,000, $40,000, $30,000 and $20,000 at the ends of
years 1 to 4. At 9 %, each cash flow is divided by 1.09 once for every year it is away, and the results are added: a
net present value of $16,872.77, the number the video worked out by hand.

In [ ]:
flows = np.array([50000, 40000, 30000, 20000])
years = np.array([1, 2, 3, 4])
line_npv = -100000 + np.sum(flows / 1.09 ** years)
print(f"NPV = {line_npv:,.2f}")

In [ ]:
# The same in one function, for any project: flows[k] arrives at time first + k.
def npv(flows, r, first=0):
    t = first + np.arange(len(flows))
    return float(np.sum(np.asarray(flows, float) / (1 + r) ** t))

line = [-100000, 50000, 40000, 30000, 20000]
right_npv = npv(line, 0.09)                              # the cost is today's, at time 0
wrong_npv = npv(line, 0.09, first=1)                     # a spreadsheet's NPV with the cost inside: everything a year late
print(f"first flow today: {right_npv:,.2f}   pushed a year late: {wrong_npv:,.2f} = {right_npv:,.2f} / 1.09")
assert round(float(line_npv), 2) == round(right_npv, 2) == 16872.77 and round(wrong_npv, 2) == 15479.60   # the video's numbers
assert round(npv(flows, 0.09, first=1), 2) == 116872.77 and abs(wrong_npv - right_npv / 1.09) < 1e-9

### How you'd do this in Python

Python has no built-in NPV or IRR, and doesn't need one: each rule from the course is a few lines. The cell below
writes them once (the comment beside each names the spreadsheet function that does the same job), then computes
every warm-up answer and checks it against the practice set. The internal rate of return has no formula: a
spreadsheet's IRR tries rates until the NPV is zero, and `irr` below does the same by halving an interval.

In [ ]:
def pv(amount, r, t): return amount / (1 + r) ** t                 # PV(r, t, 0, -amount, 0)

def solve(f, lo, hi):
    """The x between lo and hi where f(x) = 0, by halving the interval (f must change sign between them)."""
    f_lo = f(lo)
    for _ in range(200):
        mid = (lo + hi) / 2
        if (f(mid) > 0) == (f_lo > 0): lo, f_lo = mid, f(mid)
        else: hi = mid
    return (lo + hi) / 2

def irr(flows, lo=-0.9, hi=5.0):                                   # IRR(row): the rate at which NPV is zero
    return solve(lambda r: npv(flows, r), lo, hi)

def payback(flows):                                                # whole years until the later cash flows repay today's outlay
    flows, total = list(flows), 0.0
    for year, cash in enumerate(flows[1:], 1):
        total += cash
        if total >= -flows[0] - 1e-9: return year
    return None                                                    # it never pays back

def discounted_payback(flows, r):                                  # the same count, on present values
    return payback([c / (1 + r) ** t for t, c in enumerate(flows)])

def book_return(flows):                                            # average book income ÷ average book value
    cost, life = -flows[0], len(flows) - 1                         # straight-line depreciation to zero over the life
    income = [c - cost / life for c in flows[1:]]
    return (sum(income) / life) / (cost / 2)                       # straight-line: the average book value is half the cost

def op_cash_flow(revenue, costs, depreciation, tax):               # (revenue − costs) after tax, plus the tax shield
    return (revenue - costs) * (1 - tax) + tax * depreciation

def real_rate(nominal, inflation): return (1 + nominal) / (1 + inflation) - 1
def annuity_factor(r, t): return (1 - (1 + r) ** -t) / r          # PV(r, t, -1, 0, 0)
def annuity(c, r, t): return c * annuity_factor(r, t)              # PV(r, t, -c, 0, 0)
def eac(price, running, r, life):                                  # −PMT(r, life, price, 0, 0) + running
    return price / annuity_factor(r, life) + running

p11 = pv(20000, 0.09, 3)                                           # P1.1
p12 = npv([-5000, 3000, 3000], 0.09)                               # P1.2
p13 = payback([-12000] + [4000] * 5)                               # P1.3
p14 = book_return([-60000, 26000, 26000, 26000])                   # P1.4
p15 = irr([-40000, 46000])                                         # P1.5
advance = [10000, -11500]
p16 = 2 if advance[0] > 0 else 1                                   # P1.6: cash in first, out later, is borrowing
p17 = 20000 / 80000                                                # P1.7: NPV ÷ investment
p18 = op_cash_flow(200000, 140000, 30000, 0.25)                    # P1.8
p19 = real_rate(0.09, 0.03)                                        # P1.9
p110 = eac(20000, 3000, 0.06, 4)                                   # P1.10

print(f"PV {p11:,.2f}   NPV {p12:,.2f}   payback {p13} years   book return {p14:.1%}   IRR {p15:.2%}")
print(f"lending or borrowing: type {p16}   index {p17:.2f}   operating cash flow {p18:,.2f}   real rate {p19:.4%}   "
      f"equivalent annual cost {p110:,.2f}")
for k, v in (("P1.1", [p11]), ("P1.2", [p12]), ("P1.3", [p13]), ("P1.4", [100 * p14]), ("P1.5", [100 * p15]),
             ("P1.6", [p16]), ("P1.7", [p17]), ("P1.8", [p18]), ("P1.9", [100 * p19]), ("P1.10", [p110])):
    assert agrees(k, v), k
# the sorting line, as the video has it: payback 3 years, book return 20 %, IRR 17.80 %, and NPV zero at that rate
assert payback(line) == 3 and discounted_payback(line, 0.09) == 3 and abs(book_return(line) - 0.20) < 1e-12
assert round(100 * irr(line), 2) == 17.80 and abs(npv(line, irr(line))) < 1e-6

### Slide the cost of capital

The curve is the sorting line's NPV at every rate. At 0 % nothing is discounted and the NPV is the plain sum,
$40,000. At 9 % it is $16,872.77. Keep sliding: the curve crosses zero at 17.80 % (the VIOLET line), which is the
internal rate of return, and beyond it the project destroys value. The cash flows never changed; only the return
investors give up elsewhere did.

In [ ]:
def curve_picture(project, rate_pct, top, name, crossings=None, low=0.0):
    """A project's NPV against the rate (0 to top %), the slider's rate marked, the curve's zero crossings in VIOLET."""
    rates = np.linspace(low, top / 100, 300)
    values = np.array([npv(project, r) for r in rates])
    here = npv(project, rate_pct / 100)
    fig, ax = plt.subplots(figsize=(11.5, 4.6))
    fig.subplots_adjust(left=0.1, right=0.98, top=0.84, bottom=0.14)
    ax.plot(100 * rates, values, color=SKY)
    ax.axhline(0, color=DIM, lw=1)
    ax.scatter([rate_pct], [here], s=110, color=GOLD if here >= 0 else CORAL, zorder=4)
    ax.set_xlim(100 * low, top); ax.set_xlabel("cost of capital (% a year)"); ax.set_ylabel("NPV ($)")
    for x in (crossings if crossings is not None else [irr(project)]):
        ax.axvline(100 * x, color=VIOLET, ls="--", lw=1.8)
        ax.text(100 * x + 0.3, ax.get_ylim()[0], f" IRR {100 * x:.2f} %", color=VIOLET, ha="left", va="bottom", fontsize=12)
    fig.suptitle(f"{name} at {rate_pct:g} %: NPV = {here:,.2f}".replace("-", "−"), x=0.1, ha="left", fontsize=15,
                 color=GOLD if here >= 0 else CORAL)
    footnote(fig); plt.show()

def line_picture(rate_pct=9.0):
    curve_picture(line, rate_pct, 30, "the sorting line")

assert npv(line, 0.0) == 40000 and npv(line, 0.17) > 0 > npv(line, 0.18)

In [ ]:
interact(line_picture, rate_pct=FloatSlider(value=9, min=0, max=30, step=0.5, description="rate (%)"));

## Tier 2 · Core

The module's standard problems, several steps each: a net present value and the spreadsheet trap beside it, payback
and discounted payback, a book rate of return, an internal rate of return, a project with two of them, two projects
of different size, a budget to share, a machine's cash flows after tax, and two machines with different lives.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

**A project with two rates of return.** `irr` finds one crossing between two rates. When the signs of the cash flows
change twice, the NPV curve can cross zero twice, so `all_irrs` works the NPV out on a grid of rates with numpy and
refines every change of sign it meets. **A limited budget**: `by_index` ranks the projects by profitability index
(NPV ÷ investment) and takes each one that still fits. Each line below computes one core answer and checks it
against the practice set.

In [ ]:
def all_irrs(flows, lo=-0.5, hi=2.0, steps=2500):
    """Every rate between lo and hi at which NPV is zero: NPV on a grid of rates, each change of sign refined."""
    rates = np.linspace(lo, hi, steps + 1)
    values = np.array([npv(flows, r) for r in rates])
    found = []
    for a, b, f_a, f_b in zip(rates, rates[1:], values, values[1:]):
        if (f_a > 0) != (f_b > 0):
            x = solve(lambda r: npv(flows, r), a, b)
            if not found or abs(x - found[-1]) > 1e-6: found.append(x)
    return found

def by_index(projects, budget):
    """Rank by profitability index and take each project that still fits. projects: {name: (investment, NPV)}."""
    left, taken = budget, []
    for name in sorted(projects, key=lambda p: -projects[p][1] / projects[p][0]):
        if projects[name][0] <= left: taken.append(name); left -= projects[name][0]
    return sum(projects[p][1] for p in taken), sorted(taken)

upgrade = [-45000, 15000, 18000, 22000]
p21 = npv(upgrade, 0.09)                                           # P2.1: the cost is at time 0
p22 = npv(upgrade, 0.09, first=1)                                  # P2.2: the cost pushed a year late
steady = [-50000] + [18000] * 4
p23 = [payback(steady), discounted_payback(steady, 0.09)]          # P2.3
conveyor = [-90000, 40000, 38000, 36000]
p24 = [100 * book_return(conveyor), npv(conveyor, 0.09)]           # P2.4
p25 = irr([-30000, 13000, 13000, 13000])                           # P2.5
popup = [-20000, 47000, -27300]
p26 = all_irrs(popup)                                              # P2.6: the signs change twice, so look for two rates
small, big = np.array([-20000, 27000]), np.array([-60000, 76800])
p27 = [npv(small, 0.09), npv(big, 0.09), 100 * irr(big - small)]   # P2.7: the extra project is big − small
five = {"a": (60000, 15000), "b": (50000, 14000), "c": (40000, 12000), "d": (90000, 20000), "e": (50000, 9000)}
p28, taken = by_index(five, 150000)                                # P2.8
cash = op_cash_flow(34000, 0, 80000 / 4, 0.25)                     # P2.9: the saving after tax, plus the tax shield
p29 = npv([-80000] + [cash] * 4, 0.09)
p210 = [eac(24000, 5000, 0.06, 3), eac(36000, 4500, 0.06, 5)]      # P2.10

print(f"NPV {p21:,.2f}   the one-cell NPV {p22:,.2f}   payback {p23[0]} years, discounted {p23[1]} years")
print(f"book return {p24[0]:.2f} % with an NPV of {p24[1]:,.2f}   IRR {p25:.2%}")
print(f"the pop-up store: two rates, {p26[0]:.2%} and {p26[1]:.2%}; NPV at 9 % {npv(popup, 0.09):,.2f}")
print(f"NPVs {p27[0]:,.2f} and {p27[1]:,.2f}; the extra 40,000 earns {p27[2]:.2f} %   the budget buys {p28:,.0f} "
      f"(projects {', '.join(taken)})")
print(f"the machine: {cash:,.0f} a year, NPV {p29:,.2f}   equivalent annual costs {p210[0]:,.2f} and {p210[1]:,.2f}")
for k, v in (("P2.1", [p21]), ("P2.2", [p22]), ("P2.3", p23), ("P2.4", p24), ("P2.5", [100 * p25]),
             ("P2.6", [100 * p26[0], 100 * p26[1], npv(popup, 0.09)]), ("P2.7", p27), ("P2.8", [p28]), ("P2.9", [p29]),
             ("P2.10", p210)):
    assert agrees(k, v), k
assert len(p26) == 2 and taken == ["a", "b", "c"]

A project's cash flows, built column by column. The course's packing machine costs $120,000, is depreciated
straight-line over 3 years, saves $70,000 a year before tax, ties up $9,000 of working capital while it runs, and is
sold for $12,000 at the end. Tax is 25 %. Each column is one of the things that changes because of the project; the
total is the cash flow to discount.

In [ ]:
price, life, saving, working_capital, sale, tax_rate = 120000, 3, 70000, 9000, 12000, 0.25
year = np.arange(life + 1)
machine = pd.DataFrame({"year": year}).set_index("year")
machine["saving"] = np.where(year > 0, saving, 0.0)
machine["tax on the saving"] = np.where(year > 0, -tax_rate * saving, 0.0)
machine["depreciation tax shield"] = np.where(year > 0, tax_rate * price / life, 0.0)
machine["working capital"] = np.where(year == 0, -working_capital, np.where(year == life, working_capital, 0.0))
machine["machine and its sale"] = np.where(year == 0, -price, np.where(year == life, sale * (1 - tax_rate), 0.0))
machine["total"] = machine.sum(axis=1)
machine["present value"] = machine["total"] / 1.09 ** year
print(machine.round(2).to_string())
print(f"\nNPV at 9 % = {machine['present value'].sum():,.2f}")
assert list(machine["total"]) == [-129000, 62500, 62500, 80500]
assert round(machine["present value"].sum(), 2) == round(npv(machine["total"], 0.09), 2) == 43105.22

### Slide the cost of capital: a project with two rates of return

A temporary cold store on leased land costs $100,000 to set up, earns $242,000 in year 1, and costs $145,600 to
clear away in year 2. Its signs change twice, and its NPV curve crosses zero twice: at 12 % and at 30 %. Between the
two the NPV is positive; outside them it is negative. At a 9 % cost of capital it loses about $530, though both its
rates of return are above 9 %. When a project has two rates, neither is a yardstick: go back to NPV.

In [ ]:
cold_store = [-100000, 242000, -145600]
store_rates = all_irrs(cold_store)

def store_picture(rate_pct=9.0):
    curve_picture(cold_store, rate_pct, 40, "the temporary cold store", crossings=store_rates)

assert [round(100 * x, 6) for x in store_rates] == [12, 30] and round(npv(cold_store, 0.09), 2) == -530.26
assert npv(cold_store, 0.20) > 0 > npv(cold_store, 0.40)
assert all_irrs([100000, -300000, 250000]) == []                   # a project with no rate of return at all

In [ ]:
interact(store_picture, rate_pct=FloatSlider(value=9, min=0, max=40, step=0.5, description="rate (%)"));

## Tier 3 · Stretch

Combine ideas, spot the trap, run a rule backwards: the most to pay for a project, what the payback rule turns away,
the margin of safety in an IRR, a deal that is really borrowing, two projects whose ranking depends on the cost of
capital, a budget the profitability index gets wrong, a forecast in today's dollars, and when to replace an old
machine.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

**The best use of a budget** can be found by brute force: `best_set` tries every combination of projects that the
budget can pay for (with `itertools.combinations`) and keeps the one with the most NPV. **When to replace**: price
each plan (replace now, in one year, in two, never within three) as the present value of its yearly costs, and take
the cheapest. Each line computes one stretch answer and checks it.

In [ ]:
def best_set(projects, budget):
    """Brute force: the affordable set of projects with the most NPV. projects: {name: (investment, NPV)}."""
    best = (0.0, ())
    for size in range(1, len(projects) + 1):
        for combo in combinations(sorted(projects), size):
            if sum(projects[p][0] for p in combo) <= budget:
                best = max(best, (sum(projects[p][1] for p in combo), combo))
    return best[0], list(best[1])

p31 = [annuity(12000, 0.09, 5), annuity(12000, 0.12, 5)]           # P3.1: the price at which NPV is zero
one, two = [-100000] + [40000] * 3, [-100000] + [25000] * 8
p32 = [npv(one, 0.09), npv(two, 0.09)]                             # P3.2: payback 3 years against 4
new_line = [-150000, 40000, 60000, 80000, 30000]
p33 = [100 * irr(new_line), npv(new_line, 0.09)]                   # P3.3
supplier = [80000, -30000, -30000, -30000]
p34 = [100 * irr(supplier), npv(supplier, 0.09)]                   # P3.4: borrowing, so a LOW rate is good news
fast, long_ = np.array([-100000, 70000, 40000, 20000]), np.array([-100000, 10000, 30000, 110000])
tie = irr(long_ - fast)                                            # P3.5: the curves cross where the difference has NPV 0
p35 = [100 * tie, npv(long_, 0.09) - npv(fast, 0.09)]
lumpy = {"x": (60000, 18000), "y": (50000, 14000), "z": (50000, 13000)}
p36 = [by_index(lumpy, 100000)[0], best_set(lumpy, 100000)[0]]     # P3.6
real9 = real_rate(0.09, 0.03)
p37 = [annuity(80000, real9, 5), annuity(80000, 0.09, 5)]          # P3.7: real with real; then the mix-up
old = [11000, 14000, 18000]
new_eac = eac(50000, 4000, 0.06, 7)                                # P3.8
plans = {keep: npv(old[:keep] + [new_eac] * (3 - keep), 0.06, first=1) for keep in range(4)}
p38 = [new_eac, min(plans, key=plans.get)]

print(f"the most to pay: {p31[0]:,.2f} at 9 %, {p31[1]:,.2f} at 12 %")
print(f"payback {payback(one)} and {payback(two)} years; NPVs {p32[0]:,.2f} and {p32[1]:,.2f}")
print(f"IRR {p33[0]:.2f} %, NPV {p33[1]:,.2f}: {p33[0] - 9:.2f} points of room   "
      f"the supplier's deal: IRR {p34[0]:.2f} %, NPV {p34[1]:,.2f}")
print(f"fast and long tie at {p35[0]:.2f} %; at 9 % the long one leads by {p35[1]:,.2f}")
print(f"by index {p36[0]:,.0f}, the best set {p36[1]:,.0f} (projects {', '.join(best_set(lumpy, 100000)[1])})")
print(f"savings: {p37[0]:,.2f} at the real rate {real9:.3%}; mixed up {p37[1]:,.2f}")
print(f"the new machine costs {new_eac:,.2f} a year; keep the old one {p38[1]} more year(s): plans "
      + ", ".join(f"{k}: {v:,.0f}" for k, v in plans.items()))
for k, v in (("P3.1", p31), ("P3.2", p32), ("P3.3", p33), ("P3.4", p34), ("P3.5", p35), ("P3.6", p36), ("P3.7", p37),
             ("P3.8", p38)):
    assert agrees(k, v), k
assert best_set(five, 150000)[0] == p28 and p36[0] < p36[1]        # the index works only when its picks fill the budget
assert abs(npv(fast, tie) - npv(long_, tie)) < 1e-6 and irr(fast) > irr(long_) and p35[1] > 0

### Slide the cost of capital: two projects, one choice

The fast project (SKY) returns its money early; the long one (MINT) returns more, later. The fast one has the
higher rate of return, 18.2 % against 16.6 %. But which adds more value depends on the cost of capital: below 14.42 %
(the VIOLET line, where the curves cross) the long project is ahead; above it, the fast one. At 9 % the long project
wins by about $6,034. A higher rate of return doesn't mean more value.

In [ ]:
def two_projects_picture(rate_pct=9.0):
    rates = np.linspace(0, 0.25, 300)
    fig, ax = plt.subplots(figsize=(11.5, 4.6))
    fig.subplots_adjust(left=0.1, right=0.98, top=0.84, bottom=0.14)
    ax.plot(100 * rates, [npv(fast, r) for r in rates], color=SKY, label=f"fast (IRR {irr(fast):.1%})")
    ax.plot(100 * rates, [npv(long_, r) for r in rates], color=MINT, label=f"long (IRR {irr(long_):.1%})")
    ax.axhline(0, color=DIM, lw=1); ax.axvline(100 * tie, color=VIOLET, ls="--", lw=1.8)
    ax.text(100 * tie + 0.3, 48000, f"they tie at {100 * tie:.2f} %", color=VIOLET, ha="left", va="top", fontsize=12)
    f_, l_ = npv(fast, rate_pct / 100), npv(long_, rate_pct / 100)
    ax.scatter([rate_pct, rate_pct], [f_, l_], s=110, color=[GOLD if f_ > l_ else CHALK, GOLD if l_ >= f_ else CHALK], zorder=4)
    ax.set_xlim(0, 25); ax.set_xlabel("cost of capital (% a year)"); ax.set_ylabel("NPV ($)")
    ax.legend(loc="upper right", frameon=False)
    fig.suptitle(f"at {rate_pct:g} %: fast {f_:,.0f}, long {l_:,.0f}: the {'long' if l_ >= f_ else 'fast'} project adds more"
                 .replace("-", "−"), x=0.1, ha="left", fontsize=15, color=GOLD)
    footnote(fig); plt.show()

assert round(100 * tie, 2) == 14.42 and round(100 * irr(fast), 1) == 18.2 and round(100 * irr(long_), 1) == 16.6

In [ ]:
interact(two_projects_picture, rate_pct=FloatSlider(value=9, min=0, max=25, step=0.5, description="rate (%)"));

## Tier 4 · The case: Bangalore Cold Chain's automated warehouse

Bangalore Cold Chain runs refrigerated warehouses (a fictional company; our own data). It is budgeting a new
automated warehouse on a site it already owns. Two designs fit the site, and only one can be built. All amounts are
in $ millions, and the forecasts are in today's dollars.

| | full design | compact design |
|---|---|---|
| the building's cost today | 36.0 | 12.0 |
| revenue a year | 17.0 | 7.5 |
| cash costs a year | 6.0 | 2.6 |
| margin lost at the company's older warehouse, a year | 1.0 | 0.2 |
| sale price at the end of year 6 | 8.0 | 2.5 |
| working capital at the start | 1.0 | 0.6 |

Both designs: every forecast rises with 3 % expected inflation; tax is 25 %; the building is depreciated
straight-line to zero over the six forecast years (our own simple schedule), and its sale is taxed on the whole
price; the site's present tenant pays $0.6 million a year of rent, which stops either way; working capital in years
1 to 5 is a tenth of that year's revenue, and it all comes back in year 6. The opportunity cost of capital is 9 % a
year (nominal). A $1.5 million design study is already paid, and a bank has offered a $20 million loan at 5 %.

**Which design should be built, and with which cooling plant?** The next cell builds each design's forecast as a
table, one row a year (the same rows as the spreadsheet's Case study tab), and checks a few of its numbers.

In [ ]:
COST_OF_CAPITAL, INFLATION, TAX = 0.09, 0.03, 0.25
RENT, WC_SHARE, YEARS = 0.6, 0.10, 6                               # the tenant's rent · working capital ÷ revenue · the forecast
FULL = {"capex": 36.0, "revenue": 17.0, "costs": 6.0, "lost": 1.0, "salvage": 8.0, "wc0": 1.0}
COMPACT = {"capex": 12.0, "revenue": 7.5, "costs": 2.6, "lost": 0.2, "salvage": 2.5, "wc0": 0.6}
STUDY, LOAN, LOAN_RATE = 1.5, 20.0, 0.05                           # sunk · financing: neither is in the cash flows
PLANTS = {"long-life": (4.0, 10, 0.25), "short-life": (2.4, 5, 0.35)}   # price, life, running cost a year (after tax, today's dollars)

def forecast(design, inflation=INFLATION):
    """One design's forecast, year by year, in $ millions: a DataFrame with a row for each year, 0 to 6."""
    year = np.arange(YEARS + 1)
    prices = (1 + inflation) ** year
    running = year > 0
    t = pd.DataFrame({"year": year}).set_index("year")
    t["revenue"] = np.where(running, design["revenue"] * prices, 0.0)
    t["costs"] = np.where(running, design["costs"] * prices, 0.0)
    t["lost margin"] = np.where(running, design["lost"] * prices, 0.0)
    t["lost rent"] = np.where(running, RENT * prices, 0.0)
    t["depreciation"] = np.where(running, design["capex"] / YEARS, 0.0)
    t["taxable profit"] = t["revenue"] - t["costs"] - t["lost margin"] - t["lost rent"] - t["depreciation"]
    t["tax"] = TAX * t["taxable profit"]
    t["operating cash flow"] = t["taxable profit"] - t["tax"] + t["depreciation"]
    held = np.where(year == 0, design["wc0"], np.where(year == YEARS, 0.0, WC_SHARE * t["revenue"]))
    t["working capital"] = held
    t["working capital flow"] = -np.diff(held, prepend=0.0)        # money tied up is a cash flow out; released, a cash flow in
    t["building and sale"] = np.where(year == 0, -design["capex"],
                                      np.where(year == YEARS, design["salvage"] * prices * (1 - TAX), 0.0))
    t["total"] = t["operating cash flow"] + t["working capital flow"] + t["building and sale"]
    return t

full, compact = forecast(FULL), forecast(COMPACT)

# The case's numbers as the course computed them, to check these tables against.
# @STATS
CASE_STATS = {'F_npv': 9.182358671658857,
 'C_npv': 6.47042061978822,
 'F_irr': 0.15981676081671592,
 'C_irr': 0.23094665309834672,
 'F_payback': 5,
 'C_payback': 4,
 'F_dpayback': 6,
 'C_dpayback': 5,
 'F_pi': 0.24817185599077993,
 'C_pi': 0.5135254460149381,
 'step_npv': 2.7119380518706393,
 'step_irr': 0.12173062005816682,
 'gain': 2.7119380518706375,
 'F_pre1': 9.682,
 'F_dep': 6.0,
 'F_tax1': 0.9205000000000005,
 'F_ocf1': 8.761500000000002,
 'F_shield': 1.5,
 'F_shield_pv': 6.728877885346404,
 'F_salvage': 7.164313779174001,
 'F_wc_pv': -0.6766671887987794,
 'F_sunk': 7.682358671658857,
 'F_no_rent': 11.407356655642637,
 'F_no_lost': 12.890688644965156,
 'F_naive': 15.115686628948934,
 'interest': 1.0,
 'F_interest': 5.817919728985656,
 'real': 0.058252427184465994,
 'F_npv_real': 9.182358671658868,
 'F_mixed': 5.303660528582046,
 'AF10': 7.421410895582398,
 'AF5': 4.232465593403066,
 'long_eac': 0.7889810719658445,
 'short_eac': 0.9170453656470973,
 'plant_gain': 0.12806429368125272,
 'long_tie': 4.950417744461112,
 'tie_rate': 0.12173062005816682,
 'F_rev_even': 14.500299139122127}
# @END STATS
S = CASE_STATS
assert len(full) == len(compact) == 7 and abs(full["working capital flow"].sum()) < 1e-12
assert [round(v, 2) for v in full["total"]] == [-37.0, 8.01, 8.93, 9.15, 9.38, 9.62, 19.05]
assert [round(v, 2) for v in compact["total"]] == [-12.6, 3.49, 3.74, 3.84, 3.94, 4.04, 7.28]
print(f"{len(full)} rows (years 0 to 6), {full.shape[1]} columns: the full design's forecast, $ millions.")
full[["revenue", "costs", "taxable profit", "tax", "operating cash flow", "working capital flow", "building and sale",
      "total"]].round(2)

### How you'd do this in Python

With a forecast in a DataFrame, valuing a design is its `total` column through `npv`. The cell below reproduces the
video's case: year 1 of the full design, what working capital costs, the NPV in nominal and in real terms, the
mix-up that hides value, then the compact design and the three rules that prefer it, and the extra project that
settles the choice.

In [ ]:
pre_tax_1 = full.loc[1, "revenue"] - full.loc[1, "costs"] - full.loc[1, "lost margin"] - full.loc[1, "lost rent"]
wc_value = npv(full["working capital flow"], COST_OF_CAPITAL)                             # it all comes back, and still costs
full_npv, compact_npv = npv(full["total"], COST_OF_CAPITAL), npv(compact["total"], COST_OF_CAPITAL)
full_irr, compact_irr = irr(full["total"]), irr(compact["total"])
real_cc = real_rate(COST_OF_CAPITAL, INFLATION)
todays_dollars = full["total"] / (1 + INFLATION) ** full.index                            # every cash flow in today's dollars
full_real = npv(todays_dollars, real_cc)                                                  # real with real: the same NPV
full_mixed = npv(forecast(FULL, inflation=0.0)["total"], COST_OF_CAPITAL)                 # the trap: today's prices at 9 %
extra = full["total"] - compact["total"]                                                  # the extra project: full − compact
extra_npv, extra_irr = npv(extra, COST_OF_CAPITAL), irr(extra)

print(f"1 · year 1, full design: before tax {pre_tax_1:.2f}, tax {full.loc[1, 'tax']:.2f}, operating cash flow "
      f"{full.loc[1, 'operating cash flow']:.2f}; working capital costs {wc_value:.2f} in present value")
print("2 · the full design's cash flows: " + ", ".join(f"{v:.2f}" for v in full["total"]))
print(f"3 · NPV at 9 %: {full_npv:.2f}; in real terms at {real_cc:.3%}: {full_real:.2f}, the same; left in today's "
      f"prices at 9 %: {full_mixed:.2f}")
print(f"4 · the compact design: NPV {compact_npv:.2f}. IRR {compact_irr:.2%} against {full_irr:.2%}; payback "
      f"{payback(compact['total'])} years against {payback(full['total'])}; value per dollar "
      f"{compact_npv / -compact['total'][0]:.2f} against {full_npv / -full['total'][0]:.2f}")
print(f"5 · the extra {-extra[0]:.1f} million of the full design earns {extra_irr:.2%}: NPV {extra_npv:.2f}. "
      f"The full design adds {full_npv - compact_npv:.2f} more.")
assert abs(pre_tax_1 - S["F_pre1"]) < 1e-9 and abs(full.loc[1, "tax"] - S["F_tax1"]) < 1e-9
assert abs(full.loc[1, "operating cash flow"] - S["F_ocf1"]) < 1e-9 and abs(wc_value - S["F_wc_pv"]) < 1e-9
assert abs(full_npv - S["F_npv"]) < 1e-9 and abs(compact_npv - S["C_npv"]) < 1e-9 and abs(full_real - S["F_npv_real"]) < 1e-9
assert abs(full_irr - S["F_irr"]) < 1e-9 and abs(compact_irr - S["C_irr"]) < 1e-9 and abs(full_mixed - S["F_mixed"]) < 1e-9
assert abs(extra_npv - S["step_npv"]) < 1e-9 and abs(extra_irr - S["step_irr"]) < 1e-9 and abs(real_cc - S["real"]) < 1e-12
assert (payback(full["total"]), payback(compact["total"])) == (S["F_payback"], S["C_payback"]) == (5, 4)
assert [round(v, 2) for v in (full_npv, compact_npv, full_mixed, extra_npv)] == [9.18, 6.47, 5.30, 2.71]
assert [round(100 * v, 2) for v in (full_irr, compact_irr, extra_irr)] == [15.98, 23.09, 12.17]

### Slide the cost of capital: which design wins?

The full design (GOLD) and the compact design (SKY), each valued at every cost of capital. The compact design has
the higher rate of return, yet at 9 % the full design adds more. The curves cross at 12.17 % (the VIOLET line): the
full design is the better choice while the cost of capital stays below that.

In [ ]:
def designs_picture(rate_pct=9.0):
    rates = np.linspace(0, 0.30, 300)
    fig, ax = plt.subplots(figsize=(11.5, 4.6))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14)
    ax.plot(100 * rates, [npv(full["total"], r) for r in rates], color=GOLD, label=f"full design (IRR {full_irr:.1%})")
    ax.plot(100 * rates, [npv(compact["total"], r) for r in rates], color=SKY, label=f"compact design (IRR {compact_irr:.1%})")
    ax.axhline(0, color=DIM, lw=1); ax.axvline(100 * extra_irr, color=VIOLET, ls="--", lw=1.8)
    ax.text(100 * extra_irr + 0.3, 26, f"they tie at {100 * extra_irr:.2f} %", color=VIOLET, ha="left", va="top", fontsize=12)
    f_, c_ = npv(full["total"], rate_pct / 100), npv(compact["total"], rate_pct / 100)
    ax.scatter([rate_pct, rate_pct], [f_, c_], s=110, color=CHALK, zorder=4)
    ax.set_xlim(0, 30); ax.set_xlabel("cost of capital (% a year)"); ax.set_ylabel("NPV ($ millions)")
    ax.legend(loc="upper right", frameon=False)
    fig.suptitle(f"at {rate_pct:g} %: full {f_:.2f}, compact {c_:.2f}: build the {'full' if f_ >= c_ else 'compact'} design"
                 .replace("-", "−"), x=0.08, ha="left", fontsize=15, color=GOLD)
    footnote(fig, "a fictional company · our own data · computed"); plt.show()

assert abs(npv(full["total"], extra_irr) - npv(compact["total"], extra_irr)) < 1e-9

In [ ]:
interact(designs_picture, rate_pct=FloatSlider(value=9, min=0, max=30, step=0.5, description="rate (%)"));

### Your turn: tier 4

Reproduce the video's case numbers, then go further: the cost of capital at which the two designs tie, the working
capital valued on its own, the cooling plant, and your own recommendation with the numbers that anchor it. Use the
DataFrames `full` and `compact` and the functions above in a cell of your own (press **+** to add one), then type
your answers, in $ millions to two decimals.

In [ ]:
practice("P4.1")

In [ ]:
practice("P4.2")

In [ ]:
practice("P4.3")

In [ ]:
practice("P4.4")

In [ ]:
practice("P4.5")

In [ ]:
practice("P4.6")

In [ ]:
practice("P4.7")

In [ ]:
practice("P4.8")

### The tier 4 answers, from the DataFrames

Try the questions first. This cell works every tier 4 answer out from `full`, `compact` and the functions above and
checks it against the practice set: open it when you've finished, or when you're stuck.

In [ ]:
mine = {}
mine["P4.1"] = [RENT + FULL["lost"]]                               # the rent given up and the margin lost; not the study, not the interest
mine["P4.2"] = [pre_tax_1, full.loc[1, "tax"], full.loc[1, "operating cash flow"]]
mine["P4.3"] = [full.loc[1, "working capital flow"], wc_value]
mine["P4.4"] = [full_npv, 100 * full_irr]
mine["P4.5"] = [compact_npv, 100 * compact_irr, payback(full["total"]), payback(compact["total"])]
mine["P4.6"] = [extra_npv, 100 * extra_irr]
mine["P4.7"] = [full_real, full_mixed]
plant_cost = {name: eac(cost, run, real_cc, life) for name, (cost, life, run) in PLANTS.items()}   # today's dollars: the real rate
mine["P4.8"] = [plant_cost["long-life"], plant_cost["short-life"]]

for k, got in mine.items():
    got = [float(v) for v in got]
    assert agrees(k, got), (k, got, answers(k))
    print(f"{k}: " + " · ".join(f"{p}: {show(round(v, 4))}" for p, v in zip(Q[k]["parts"], got)))
assert abs(plant_cost["long-life"] - S["long_eac"]) < 1e-9 and abs(plant_cost["short-life"] - S["short_eac"]) < 1e-9
print("\nEvery tier 4 answer worked out from the DataFrames matches the practice set.")
print(f"Build the full design: NPV {full_npv:.2f} million, {full_npv - compact_npv:.2f} more than the compact design, "
      f"while the cost of capital stays below {100 * extra_irr:.1f} %. Use the long-life plant: it costs "
      f"{plant_cost['short-life'] - plant_cost['long-life']:.2f} million a year less to own.")

## Sources
Every exercise, example and company here is our own (Bangalore Cold Chain is fictional). The order of topics follows
the capital-budgeting chapters of a standard corporate finance course (why net present value leads to better
decisions than other rules, then making investment decisions with it); the rules are the standard ones: net present
value, payback, the book rate of return, the internal rate of return, the profitability index, cash flows after tax,
and equivalent annual cost. The spreadsheet functions named in the hints (NPV, IRR, PMT, PV, SUM, SUMPRODUCT,
AVERAGE) have the same names and take their arguments in the same order in the two most widely used spreadsheets
(their official function lists, retrieved 2026-11-11).

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib
_src = pathlib.Path("../../../episodes/cf03_npv_decisions/practice.py")
if _src.exists():
    _spec = importlib.util.spec_from_file_location("cf03_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"]) == (_e["q"], _e["working"], _e["function"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])) and all(
            abs(a - float(b)) < 1e-9 for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
        _t = _e["tol"] if isinstance(_e["tol"], tuple) else (_e["tol"],) * len(_q["answer"])
        assert list(_t) == _q["tol"], _e["id"]
    assert len(_P.EX) == len(EX)
    assert all(abs(float(_P.CS[k]) - v) < 1e-9 for k, v in CASE_STATS.items()) and set(_P.CS) == set(CASE_STATS)
    _C = _P.CASE
    assert (_C["r"], _C["inflation"], _C["tax"], _C["rent"], _C["wc_share"], _C["years"]) == (
        COST_OF_CAPITAL, INFLATION, TAX, RENT, WC_SHARE, YEARS)
    assert _C["full"] == FULL and _C["compact"] == COMPACT and _C["study"] == STUDY
    assert (_C["loan"]["amount"], _C["loan"]["rate"]) == (LOAN, LOAN_RATE)
    assert {k: (v["cost"], v["life"], v["run"]) for k, v in _C["plant"].items()} == {
        "long": PLANTS["long-life"], "short": PLANTS["short-life"]}
    _rows = {"revenue": "revenue", "costs": "costs", "lost": "lost margin", "rent": "lost rent", "dep": "depreciation",
             "profit": "taxable profit", "tax": "tax", "ocf": "operating cash flow", "wc": "working capital",
             "dwc": "working capital flow", "capital": "building and sale", "total": "total"}
    for _mine, _theirs in ((full, _P.FULL), (compact, _P.COMPACT)):
        assert set(_theirs) == set(_rows)
        assert all(abs(a - b) < 1e-9 for k, c in _rows.items() for a, b in zip(_mine[c], _theirs[k]))
    assert line == _P.LINE and cold_store == _P.TWO and list(fast) == _P.FAST and list(long_) == _P.LONG
    assert abs(tie - _P.CROSS) < 1e-9 and abs(irr(line) - _P.LINE_IRR) < 1e-9 and lumpy == _P.LUMPY
    assert list(machine["total"]) == _P.Q12_FLOWS and abs(npv(machine["total"], 0.09) - _P.Q12_NPV) < 1e-9
    print("matches the course's practice set")